# headswap_V2 - upload your own pair and test

**Run order:** Cell 1 (setup, restarts the kernel) -> Cell 2 (upload) -> Cell 3 (run).

Cell 1 only needs re-running if the runtime is recycled. To try another pair, just re-run Cells 2 and 3.

The pipeline auto-routes: a visible body goes through the full-body path (head swap + donor skin tone on exposed skin); a tight head shot uses crop_stitch.


In [ ]:
#@title Cell 1 - Setup (run once, restarts the kernel automatically)
from pathlib import Path
import subprocess, shutil, os, signal

assert Path("/content").exists(), "Open this notebook in Google Colab."
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run this cell.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
      subprocess.getoutput(f"git -C {REPO} log -1 --pretty=%s"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

if not Path("/content/ComfyUI/server.py").exists():
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("--- stderr ---"); print(r.stderr[-2000:])
        raise SystemExit("setup_colab.sh failed")
else:
    print("ComfyUI already present - skipping setup")

subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "--no-deps", "numpy==2.4.6"], check=True)

print("\n\u2713 Setup complete. Restarting kernel (this is expected)...")
print("   When it comes back, run Cell 2.")
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
#@title Cell 2 - Upload YOUR two images
# Upload the BODY first (the photo you want to keep: pose, clothes, background),
# then the FACE (the person whose identity + skin tone you want transferred in).
import os
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "my_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def _grab(role):
    print(f"\n=== Upload the {role} image ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} image uploaded - re-run this cell.")
    name = next(iter(up))
    dest = PAIR / f"{role}.png"
    Image.open(name).convert("RGB").save(dest)
    os.remove(name)
    im = Image.open(dest)
    print(f"saved {role}: {im.size[0]}x{im.size[1]}px")
    if max(im.size) < 500:
        print(f"   note: small source ({im.size[0]}x{im.size[1]}). The pipeline "
              "upscales to 1024 so generated detail survives, but a larger "
              "original will always look sharper.")
    return im

body_im = _grab("body")
face_im = _grab("face")

print("\n--- BODY (kept: pose / clothing / background) ---")
display(body_im)
print("--- FACE (donor: identity + skin tone) ---")
display(face_im)
print("\n\u2713 Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - Run the head swap
SEED = 46  #@param {type:"integer"}

import sys, os, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

body_im = Image.open(body_path).convert("RGB")
face_im = Image.open(face_path).convert("RGB")

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({"seed": int(SEED), "save_debug": False, "verbose": False})

t0 = time.perf_counter()
result = create_pipeline(cfg, runtime=runtime).run(body_im, face_im)
elapsed = time.perf_counter() - t0

meta = result.meta or {}
route = (meta.get("body_route") or {})
skin = meta.get("skin_harmonize") or {}
print(f"\n{elapsed:.0f}s  mode={meta.get('edit_mode')}  out={result.image.size}")
print(f"route={route.get('route')}  reason={route.get('reason')}")
if skin:
    print(f"skin: applied={skin.get('applied')} px={skin.get('skin_px')} "
          f"src={skin.get('src_lab_mean')} -> tgt={skin.get('tgt_lab_mean')}")

out_path = REPO / "results" / "my_pair_result.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
result.image.save(out_path)

display(Markdown("### Result"))
display(result.image)

# Uncomment to download:
# from google.colab import files; files.download(str(out_path))
